# Train the Baseline CNN on a Free GPU

This notebook downloads the weather data, trains the 3-day and 5-day forecast networks, scores them on 2017 and 2018, and downloads the results as `cnn_results.zip`.

In the menu choose Runtime > Run all. It takes roughly 30 to 60 minutes.

Progress is saved after every epoch. If the session drops, open the notebook again and choose Runtime > Run all: finished steps are skipped and training continues from the last completed epoch.

With `SAVE_TO_DRIVE = True` the progress is kept in a `weather-forecaster` folder in your Google Drive, so it survives even when Colab gives you a new machine. Colab asks for permission to use your Drive when the first cell runs. Delete that folder to train from scratch.

In [ ]:
SAVE_TO_DRIVE = True  # False keeps progress on this machine only, which a dropped session can lose

import os

import torch

assert torch.cuda.is_available(), (
    "No GPU found. In the menu choose Runtime > Change runtime type > T4 GPU, then Run all again."
)
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

if SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    KEEP = "/content/drive/MyDrive/weather-forecaster"
else:
    KEEP = "/content/weather-forecaster-progress"
os.makedirs(KEEP, exist_ok=True)
print("progress is kept in", KEEP)

In [ ]:
import json
import subprocess
import sys

REPO = "/content/weather-forecaster"
URL = "https://github.com/KairavT/weather-forecaster.git"
RESULTS = f"{KEEP}/models.json"
if not os.path.isdir(REPO):
    # the code lives on the first-model branch until its pull request is merged
    !git clone -q --branch first-model {URL} {REPO} || git clone -q {URL} {REPO}
%cd {REPO}
!pip install -q "zarr>=3" gcsfs certifi
os.environ["PYTHONPATH"] = f"{REPO}/src"


def run(*args):
    """Run one of the repo's scripts, show its output live, and stop the notebook if it fails."""
    process = subprocess.Popen(
        [sys.executable, "-u", *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise RuntimeError(f"{' '.join(args)} failed, see the output above")


def finished():
    """Scores of the networks that have already been trained."""
    if not os.path.exists(RESULTS):
        return {}
    with open(RESULTS) as f:
        return json.load(f)

Download three fields for 1979 to 2020 (about 1.5 GB) and check the no-model baselines. Anything already downloaded is skipped.

In [ ]:
run("scripts/download.py", "--preset", "baselines")
run("scripts/baselines.py")

Train one network per lead time. Each stops early once the 2016 validation score stops improving. A network that is already trained is skipped, and one that was cut off continues from its last completed epoch.

In [ ]:
for days in (3, 5):
    if f"cnn-{days}d" in finished():
        print(f"{days}-day network is already trained, skipping")
        continue
    run(
        "scripts/train.py",
        "--lead-days",
        str(days),
        "--epochs",
        "30",
        "--runs",
        f"{KEEP}/runs",
        "--results",
        RESULTS,
    )

Show the test scores next to the WeatherBench paper's CNN, then download everything as one zip.

In [ ]:
import shutil

from google.colab import files

PUBLISHED = {"cnn-3d": (626, 2.87), "cnn-5d": (757, 3.37)}  # Rasp et al. 2020, Table 2

for name, entry in finished().items():
    z500 = entry["test_rmse"]["geopotential_500"]
    t850 = entry["test_rmse"]["temperature_850"]
    paper_z500, paper_t850 = PUBLISHED.get(name, ("n/a", "n/a"))
    print(
        f"{name}: Z500 {z500:.0f} (paper {paper_z500}), T850 {t850:.2f} (paper {paper_t850}), "
        f"{entry['epochs_run']} epochs"
    )

print("\nIf the download does not start, copy this text instead:")
print(json.dumps(finished(), indent=2))
shutil.make_archive("/content/cnn_results", "zip", KEEP)
files.download("/content/cnn_results.zip")